# DL segmentation evaluation

Compute Dice similarity between nnUNet tumor segmentation and radiologist manual segmentation.

No files are saved by this notebook; results are printed in the notebook output only.

In [1]:

# ============================================================
# 1. Imports and settings
# ============================================================

from pathlib import Path

import nibabel as nb
import numpy as np

# Fixed seed so the random 20-case samples are reproducible.
random_state = 20260817
sample_n = 20

# set2: prediction folder and manual-label folder use the same set name.
set2_pred_root = Path('/host/d/projects/Habitats/segmentation/models/Dataset602_Tumor/results/predicts/set_2')
set2_manual_root = Path('/host/e/D/Data/Habitats/Jishuitan/original_data/set_2')

# set3: nnUNet prediction folder is named set_1, but it corresponds to external set3.
set3_pred_root = Path('/host/d/projects/Habitats/segmentation/models/Dataset603_TumorExternal/results/EncUNetM_3d_fullres/predicts/set_1')
set3_manual_root = Path('/host/e/D/Data/Habitats/Jishuitan/original_data/set_3')

print('random_state:', random_state)
print('sample_n:', sample_n)


random_state: 20260817
sample_n: 20


In [2]:

# ============================================================
# 2. Find cases that have prediction/manual labels with matching shapes
# ============================================================

def case_sort_key(case_id):
    case_id = str(case_id)
    return int(case_id) if case_id.isdigit() else case_id


def get_nifti_shape(path):
    return tuple(nb.load(str(path)).shape)


def find_valid_common_cases(pred_root, manual_root):
    pred_cases = {
        path.parent.name
        for path in pred_root.glob('*/pred_label.nii.gz')
    }
    manual_cases = {
        path.parent.name
        for path in manual_root.glob('*/label.nii.gz')
    }
    common_cases = sorted(pred_cases & manual_cases, key=case_sort_key)

    valid_cases = []
    shape_mismatch_cases = []

    for case_id in common_cases:
        pred_path = pred_root / str(case_id) / 'pred_label.nii.gz'
        manual_path = manual_root / str(case_id) / 'label.nii.gz'
        pred_shape = get_nifti_shape(pred_path)
        manual_shape = get_nifti_shape(manual_path)

        if pred_shape == manual_shape:
            valid_cases.append(case_id)
        else:
            shape_mismatch_cases.append((case_id, pred_shape, manual_shape))

    return pred_cases, manual_cases, common_cases, valid_cases, shape_mismatch_cases


def sample_cases(valid_cases, sample_n, random_state):
    if len(valid_cases) < sample_n:
        raise ValueError(f'Only {len(valid_cases)} valid cases found, cannot sample {sample_n}.')
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(valid_cases), size=sample_n, replace=False)
    return sorted([valid_cases[i] for i in idx], key=case_sort_key)


set2_pred_cases, set2_manual_cases, set2_common_cases, set2_valid_cases, set2_shape_mismatch_cases = find_valid_common_cases(set2_pred_root, set2_manual_root)
set3_pred_cases, set3_manual_cases, set3_common_cases, set3_valid_cases, set3_shape_mismatch_cases = find_valid_common_cases(set3_pred_root, set3_manual_root)

set2_sample_cases = sample_cases(set2_valid_cases, sample_n, random_state)
set3_sample_cases = sample_cases(set3_valid_cases, sample_n, random_state + 1)

print('set2 prediction cases:', len(set2_pred_cases))
print('set2 manual cases:', len(set2_manual_cases))
print('set2 common cases:', len(set2_common_cases))
print('set2 valid shape-matched cases:', len(set2_valid_cases))
print('set2 shape mismatches:', set2_shape_mismatch_cases)
print('set2 sampled cases:', set2_sample_cases)
print()
print('set3 prediction cases:', len(set3_pred_cases))
print('set3 manual cases:', len(set3_manual_cases))
print('set3 common cases:', len(set3_common_cases))
print('set3 valid shape-matched cases:', len(set3_valid_cases))
print('set3 shape mismatches:', set3_shape_mismatch_cases)
print('set3 sampled cases:', set3_sample_cases)


set2 prediction cases: 153
set2 manual cases: 231
set2 common cases: 150
set2 valid shape-matched cases: 149
set2 shape mismatches: [('45', (324, 321, 29), (516, 513, 29))]
set2 sampled cases: ['41', '96', '103', '110', '111', '117', '119', '151', '158', '190', '191', '192', '202', '209', '211', '232', '243', '256', '264', '290']

set3 prediction cases: 48
set3 manual cases: 21
set3 common cases: 21
set3 valid shape-matched cases: 20
set3 shape mismatches: [('28', (320, 260, 64), (512, 512, 30))]
set3 sampled cases: ['1', '2', '3', '4', '6', '9', '10', '11', '12', '14', '15', '16', '17', '19', '20', '21', '23', '24', '26', '27']


In [3]:

# ============================================================
# 3. Compute Dice for sampled cases
# ============================================================

def load_binary_mask(path):
    data = nb.load(str(path)).get_fdata()
    return data > 0


def dice_score(pred_mask, manual_mask):
    if pred_mask.shape != manual_mask.shape:
        raise ValueError(f'Shape mismatch: pred {pred_mask.shape}, manual {manual_mask.shape}')

    pred_sum = int(pred_mask.sum())
    manual_sum = int(manual_mask.sum())
    denom = pred_sum + manual_sum

    if denom == 0:
        return np.nan

    intersection = int(np.logical_and(pred_mask, manual_mask).sum())
    return 2.0 * intersection / denom


def compute_dataset_dice(dataset_name, sample_cases, pred_root, manual_root):
    dice_values = []

    print('\n============================================================')
    print(dataset_name)
    print('============================================================')

    for case_id in sample_cases:
        pred_path = pred_root / str(case_id) / 'pred_label.nii.gz'
        manual_path = manual_root / str(case_id) / 'label.nii.gz'

        pred_mask = load_binary_mask(pred_path)
        manual_mask = load_binary_mask(manual_path)
        dice = dice_score(pred_mask, manual_mask)
        dice_values.append(dice)

        print(f'case {case_id}: Dice = {dice:.4f}')

    dice_values = np.asarray(dice_values, dtype=float)
    mean_dice = float(np.nanmean(dice_values))
    std_dice = float(np.nanstd(dice_values, ddof=1))

    print('------------------------------------------------------------')
    print(f'{dataset_name} Dice mean +- std: {mean_dice:.4f} +- {std_dice:.4f}')

    return dice_values, mean_dice, std_dice


set2_dice_values, set2_mean_dice, set2_std_dice = compute_dataset_dice(
    'set2', set2_sample_cases, set2_pred_root, set2_manual_root,
)

set3_dice_values, set3_mean_dice, set3_std_dice = compute_dataset_dice(
    'set3', set3_sample_cases, set3_pred_root, set3_manual_root,
)

print('\n============================================================')
print('Final summary')
print('============================================================')
print(f'set2: {set2_mean_dice:.4f} +- {set2_std_dice:.4f}')
print(f'set3: {set3_mean_dice:.4f} +- {set3_std_dice:.4f}')



set2
case 41: Dice = 0.9981


case 96: Dice = 0.9417
case 103: Dice = 1.0000


case 110: Dice = 0.9553
case 111: Dice = 0.9964


case 117: Dice = 0.9921


case 119: Dice = 0.9921
case 151: Dice = 0.9906


...

case 6: Dice = 0.9538
case 9: Dice = 0.9294
case 10: Dice = 0.9461


case 11: Dice = 0.0000
case 12: Dice = 1.0000
case 14: Dice = 0.9639
case 15: Dice = 0.9695


case 16: Dice = 0.9684
case 17: Dice = 1.0000
case 19: Dice = 0.9960


case 20: Dice = 0.0000
case 21: Dice = 1.0000
case 23: Dice = 0.9948
case 24: Dice = 0.8528


case 26: Dice = 1.0000
case 27: Dice = 0.9772
------------------------------------------------------------
set3 Dice mean +- std: 0.8766 +- 0.3019

Final summary
set2: 0.9057 +- 0.2579
set3: 0.8766 +- 0.3019
